In [39]:
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from scipy.interpolate import griddata
from scipy.interpolate import Rbf
from itertools import combinations
from scipy.optimize import minimize

In [44]:
X3_obs = np.load("f3initial_inputs.npy")
y3_obs = np.load("f3initial_outputs.npy")
print("file shapes F3 ", X3_obs.shape, y3_obs.shape)

X4_obs = np.load("f4initial_inputs.npy")
y4_obs = np.load("f4initial_outputs.npy")
print("file shapes F4 ", X4_obs.shape, y4_obs.shape)

X5_obs = np.load("f5initial_inputs.npy")
y5_obs = np.load("f5initial_outputs.npy")
print("file shapes F5 ", X5_obs.shape, y5_obs.shape)

X6_obs = np.load("f6initial_inputs.npy")
y6_obs = np.load("f6initial_outputs.npy")
print("file shapes F6 ", X6_obs.shape, y6_obs.shape)

X7_obs = np.load("f7initial_inputs.npy")
y7_obs = np.load("f7initial_outputs.npy")
print("file shapes F7 ", X7_obs.shape, y7_obs.shape)

X8_obs = np.load("f8initial_inputs.npy")
y8_obs = np.load("f8initial_outputs.npy")
print("file shapes F8 ", X8_obs.shape, y8_obs.shape)


file shapes F3  (18, 3) (18,)
file shapes F4  (33, 4) (33,)
file shapes F5  (23, 4) (23,)
file shapes F6  (23, 5) (23,)
file shapes F7  (33, 6) (33,)
file shapes F8  (43, 8) (43,)


In [45]:
# F3 Load data, fit GP and find the next highest UCB point
# Load data
X3_obs = np.load("f3initial_inputs.npy")
y3_obs = np.load("f3initial_outputs.npy")
bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
kappa = 4.0

X_obs = np.atleast_2d(X3_obs)
y_obs = np.asarray(y3_obs).ravel()
N, dim = X_obs.shape
assert len(bounds) == dim, "Bounds must match dimensionality"

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0, length_scale_bounds=(0.01, 10.0))
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,  # Reduced noise assumption
    n_restarts_optimizer=10,  # Better kernel optimization
    normalize_y=True
)

gp.fit(X_obs, y_obs)

n_grid = 50
x0_grid = np.linspace(0, 1, n_grid)
x1_grid = np.linspace(0, 1, n_grid)
x2_grid = np.linspace(0, 1, n_grid)

X_grid = np.array(np.meshgrid(x0_grid, x1_grid, x2_grid)).T.reshape(-1, 3)

mu_grid, std_grid = gp.predict(X_grid, return_std=True)
ucb_grid = mu_grid + kappa * std_grid

best_ucb_idx = np.argmax(ucb_grid)
next_point = X_grid[best_ucb_idx:best_ucb_idx+1]

print(f"Next F3 point to sample: {np.round(next_point[0],6)}")
print(f"Expected UCB: {ucb_grid[best_ucb_idx]:.4f}")
print(f"Expected mean: {mu_grid[best_ucb_idx]:.4f}")
print(f"Uncertainty: {std_grid[best_ucb_idx]:.4f}")


Next F3 point to sample: [0.591837 0.836735 0.734694]
Expected UCB: 0.2677
Expected mean: -0.0852
Uncertainty: 0.0882


In [28]:
# Function to create 2d sliced plots for F3 - F8 
def plot_2d_slices(gp, X_obs, y_obs, bounds, kappa=3.0, n_grid=50):
    """
    Plot 2D slices through the GP posterior at the best observed point.
    
    Args:
        gp: Fitted GaussianProcessRegressor
        X_obs: Observed inputs (N, D)
        y_obs: Observed outputs (N,)
        bounds: List of (min, max) for each dimension
        kappa: UCB exploration parameter
        n_grid: Grid resolution
    """
    dim = len(bounds)
    
    # Find best observed point
    best_idx = np.argmax(y_obs)
    best_point = X_obs[best_idx]
    
    print(f"Slicing at best point: {best_point}")
    print(f"Best value: {y_obs[best_idx]:.4f}")
    
    # Plot all pairs of dimensions
    n_pairs = dim * (dim - 1) // 2
    n_cols = min(3, n_pairs)
    n_rows = (n_pairs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols * 3, figsize=(6 * n_cols, 5 * n_rows))
    if n_pairs == 1:
        axes = axes.reshape(1, -1)
    
    pair_idx = 0
    for i, j in combinations(range(dim), 2):
        row = pair_idx // n_cols
        col = (pair_idx % n_cols) * 3
        
        # Create grid for these two dimensions
        x_i = np.linspace(bounds[i][0], bounds[i][1], n_grid)
        x_j = np.linspace(bounds[j][0], bounds[j][1], n_grid)
        Xi, Xj = np.meshgrid(x_i, x_j)
        
        # Fix other dimensions at best point values
        X_slice = np.tile(best_point, (n_grid * n_grid, 1))
        X_slice[:, i] = Xi.ravel()
        X_slice[:, j] = Xj.ravel()
        
        # Predict
        mu, std = gp.predict(X_slice, return_std=True)
        mu = mu.reshape(n_grid, n_grid)
        std = std.reshape(n_grid, n_grid)
        ucb = mu + kappa * std
        
        # Plot Mean
        ax_mean = axes[row, col] if n_rows > 1 else axes[col]
        im1 = ax_mean.contourf(Xi, Xj, mu, levels=20, cmap='viridis')
        ax_mean.scatter(X_obs[:, i], X_obs[:, j], c='red', marker='x', s=50, linewidths=2)
        ax_mean.scatter(best_point[i], best_point[j], c='yellow', marker='*', s=300, edgecolors='black')
        ax_mean.set_xlabel(f'x{i}')
        ax_mean.set_ylabel(f'x{j}')
        ax_mean.set_title(f'Mean (x{i} vs x{j})')
        plt.colorbar(im1, ax=ax_mean)
        
        # Plot Uncertainty
        ax_std = axes[row, col + 1] if n_rows > 1 else axes[col + 1]
        im2 = ax_std.contourf(Xi, Xj, std, levels=20, cmap='plasma')
        ax_std.scatter(X_obs[:, i], X_obs[:, j], c='red', marker='x', s=50, linewidths=2)
        ax_std.scatter(best_point[i], best_point[j], c='yellow', marker='*', s=300, edgecolors='black')
        ax_std.set_xlabel(f'x{i}')
        ax_std.set_ylabel(f'x{j}')
        ax_std.set_title(f'Uncertainty (x{i} vs x{j})')
        plt.colorbar(im2, ax=ax_std)
        
        # Plot UCB
        ax_ucb = axes[row, col + 2] if n_rows > 1 else axes[col + 2]
        im3 = ax_ucb.contourf(Xi, Xj, ucb, levels=20, cmap='coolwarm')
        ax_ucb.scatter(X_obs[:, i], X_obs[:, j], c='red', marker='x', s=50, linewidths=2)
        ax_ucb.scatter(best_point[i], best_point[j], c='yellow', marker='*', s=300, edgecolors='black')
        ax_ucb.set_xlabel(f'x{i}')
        ax_ucb.set_ylabel(f'x{j}')
        ax_ucb.set_title(f'UCB (x{i} vs x{j})')
        plt.colorbar(im3, ax=ax_ucb)
        
        pair_idx += 1
    
    # Remove empty subplots
    total_plots = n_rows * n_cols * 3
    for idx in range(pair_idx * 3, total_plots):
        row = idx // (n_cols * 3)
        col = idx % (n_cols * 3)
        fig.delaxes(axes[row, col] if n_rows > 1 else axes[col])
    
    plt.tight_layout()
    plt.savefig('gp_2d_slices.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    return fig

# Usage for 3D problem:
# plot_2d_slices(gp, X_obs, y_obs, bounds=[(0,1), (0,1), (0,1)], kappa=3.0)

In [46]:
# F4 Load data, fit GP and find the next highest UCB point
# Load data
X4_obs = np.load("f4initial_inputs.npy")
y4_obs = np.load("f4initial_outputs.npy")
bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
kappa = 4.0

X_obs = np.atleast_2d(X4_obs)
y_obs = np.asarray(y4_obs).ravel()
N, dim = X_obs.shape
assert len(bounds) == dim, "Bounds must match dimensionality"

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0, length_scale_bounds=(0.01, 10.0))
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,  # Reduced noise assumption
    n_restarts_optimizer=10,  # Better kernel optimization
    normalize_y=True
)

gp.fit(X_obs, y_obs)

n_grid = 50
x0_grid = np.linspace(0, 1, n_grid)
x1_grid = np.linspace(0, 1, n_grid)
x2_grid = np.linspace(0, 1, n_grid)
x3_grid = np.linspace(0, 1, n_grid)

X_grid = np.array(np.meshgrid(x0_grid, x1_grid, x2_grid, x3_grid)).T.reshape(-1, 4)

mu_grid, std_grid = gp.predict(X_grid, return_std=True)
ucb_grid = mu_grid + kappa * std_grid

best_ucb_idx = np.argmax(ucb_grid)
next_point = X_grid[best_ucb_idx:best_ucb_idx+1]

print(f"Next F4 point to sample: {np.round(next_point[0],6)}")
print(f"Expected UCB: {ucb_grid[best_ucb_idx]:.4f}")
print(f"Expected mean: {mu_grid[best_ucb_idx]:.4f}")
print(f"Uncertainty: {std_grid[best_ucb_idx]:.4f}")


Next F4 point to sample: [0.387755 0.387755 0.387755 0.428571]
Expected UCB: 1.0805
Expected mean: -1.1490
Uncertainty: 0.5574


In [40]:
# F5 next UCB values
# ── Configuration ─────────────────────────────────────────────
# Change this number to 5, 6, 7, or 8 for F5-F8
FUNCTION_NUMBER = 5
KAPPA           = 4.0
N_RANDOM        = 100000
BATCH_SIZE      = 10000

# ── 1. Load data ──────────────────────────────────────────────
X_obs = np.atleast_2d(
    np.load(f"f{FUNCTION_NUMBER}initial_inputs.npy")
)
y_obs = np.asarray(
    np.load(f"f{FUNCTION_NUMBER}initial_outputs.npy")
).ravel()

N, dim = X_obs.shape

print("="*60)
print(f"F{FUNCTION_NUMBER}: GP FITTING AND UCB MAXIMISATION")
print("="*60)
print(f"\nData loaded:")
print(f"  Observations:  {N}")
print(f"  Dimensions:    {dim}")
print(f"  y range:       [{y_obs.min():.6f}, {y_obs.max():.6f}]")
print(f"  y mean:        {y_obs.mean():.6f}")
print(f"  y std:         {y_obs.std():.6f}")

# ── 2. Build bounds dynamically ───────────────────────────────
# Automatically creates correct number of bounds
# for whatever dimension the loaded file has
bounds = [(0.0, 1.0)] * dim
lows   = np.array([b[0] for b in bounds])
highs  = np.array([b[1] for b in bounds])

assert len(bounds) == dim, "Bounds must match dimensionality"
print(f"  Bounds:        (0.0, 1.0) × {dim} dimensions")

# ── 3. Fit GP ─────────────────────────────────────────────────
# Anisotropic RBF: separate length scale per dimension
# This is important for high-D as different dimensions
# may have very different sensitivities
kernel = ConstantKernel(
    1.0,
    constant_value_bounds=(1e-3, 1e3)
) * RBF(
    length_scale=np.ones(dim),
    length_scale_bounds=(0.01, 10.0)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    n_restarts_optimizer=10,
    normalize_y=True
)

print(f"\nFitting GP...")
gp.fit(X_obs, y_obs)
print(f"Fitted kernel: {gp.kernel_}")

# ── 4. Random search for highest UCB ──────────────────────────
# meshgrid is NOT used because:
#
# Dimension | meshgrid points (n=50) | Memory
# ──────────|───────────────────────|──────────────
#     5D    |     312,500,000        |  ~12  GB
#     6D    |  15,625,000,000        |  ~600 GB
#     7D    | 781,250,000,000        |  ~30  TB
#     8D    |         too large      |  impossible
#
# Random sampling with 100,000 points is tractable
# for any dimension and still finds good candidates

print(f"\nSearching for highest UCB...")
print(f"  Random samples: {N_RANDOM:,}")
print(f"  Batch size:     {BATCH_SIZE:,}")

np.random.seed(42)
X_candidates = np.random.uniform(
    low=lows,
    high=highs,
    size=(N_RANDOM, dim)
)

# Predict in batches to avoid memory issues
n_batches = int(np.ceil(N_RANDOM / BATCH_SIZE))
mu_all    = np.zeros(N_RANDOM)
std_all   = np.zeros(N_RANDOM)

for batch in range(n_batches):
    start = batch * BATCH_SIZE
    end   = min(start + BATCH_SIZE, N_RANDOM)
    mu_all[start:end], std_all[start:end] = gp.predict(
        X_candidates[start:end],
        return_std=True
    )

ucb_all      = mu_all + KAPPA * std_all
best_rnd_idx = np.argmax(ucb_all)
best_rnd_pt  = X_candidates[best_rnd_idx]

print(f"\nRandom search best:")
print(f"  UCB:  {ucb_all[best_rnd_idx]:.4f}")
print(f"  Mean: {mu_all[best_rnd_idx]:.4f}")
print(f"  Std:  {std_all[best_rnd_idx]:.4f}")

# ── 5. Refine with scipy optimiser ────────────────────────────
# Polishes the random search result using gradient-based
# local optimisation to find a more precise UCB maximum

def negative_ucb(x):
    x = np.atleast_2d(x)
    mu, std = gp.predict(x, return_std=True)
    return -(mu[0] + KAPPA * std[0])

print(f"\nRefining with scipy L-BFGS-B...")
result = minimize(
    negative_ucb,
    x0=best_rnd_pt,
    method='L-BFGS-B',
    bounds=bounds
)

# Use refined point only if it improves UCB
if -result.fun > ucb_all[best_rnd_idx]:
    next_point       = result.x
    mu_next, std_next = gp.predict(
        next_point.reshape(1, -1),
        return_std=True
    )
    mu_next  = mu_next[0]
    std_next = std_next[0]
    ucb_next = mu_next + KAPPA * std_next
    print(f"  ✓ Refinement improved UCB to {ucb_next:.4f}")
else:
    next_point = best_rnd_pt
    mu_next    = mu_all[best_rnd_idx]
    std_next   = std_all[best_rnd_idx]
    ucb_next   = ucb_all[best_rnd_idx]
    print(f"  ○ Random search result retained")

# ── 6. Current best observation ───────────────────────────────
best_obs_idx = np.argmax(y_obs)
best_obs_val = y_obs[best_obs_idx]
best_obs_loc = X_obs[best_obs_idx]

# ── 7. Print results ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"RESULTS: F{FUNCTION_NUMBER} ({dim}D)")
print(f"{'='*60}")

print(f"\nNext point to sample:")
for d, val in enumerate(next_point):
    print(f"  x{d}: {val:.6f}")

print(f"\nExpected UCB:        {ucb_next:.4f}")
print(f"Expected mean:       {mu_next:.4f}")
print(f"Uncertainty:         {std_next:.4f}")

print(f"\nCurrent best observation:")
print(f"  Value:             {best_obs_val:.6f}")
print(f"  Location:          {best_obs_loc}")

print(f"\nUCB vs current best: {ucb_next - best_obs_val:+.4f}")
if ucb_next > best_obs_val:
    print(f"  ✓ Next point is expected to improve on current best")
else:
    print(f"  ○ Improvement uncertain - high exploration component")

F5: GP FITTING AND UCB MAXIMISATION

Data loaded:
  Observations:  23
  Dimensions:    4
  y range:       [0.112940, 1088.859618]
  y mean:        140.390598
  y std:         231.654950
  Bounds:        (0.0, 1.0) × 4 dimensions

Fitting GP...
Fitted kernel: 1.33**2 * RBF(length_scale=[10, 8.37, 0.342, 0.125])

Searching for highest UCB...
  Random samples: 100,000
  Batch size:     10,000

Random search best:
  UCB:  1885.0273
  Mean: 991.9021
  Std:  223.2813

Refining with scipy L-BFGS-B...
  ✓ Refinement improved UCB to 1885.6313

RESULTS: F5 (4D)

Next point to sample:
  x0: 1.000000
  x1: 0.130855
  x2: 1.000000
  x3: 0.997941

Expected UCB:        1885.6313
Expected mean:       993.3081
Uncertainty:         223.0808

Current best observation:
  Value:             1088.859618
  Location:          [0.22418902 0.84648049 0.87948418 0.87851568]

UCB vs current best: +796.7717
  ✓ Next point is expected to improve on current best


In [41]:
# F6 next UCB values
# ── Configuration ─────────────────────────────────────────────
# Change this number to 5, 6, 7, or 8 for F5-F8
FUNCTION_NUMBER = 6
KAPPA           = 4.0
N_RANDOM        = 100000
BATCH_SIZE      = 10000

# ── 1. Load data ──────────────────────────────────────────────
X_obs = np.atleast_2d(
    np.load(f"f{FUNCTION_NUMBER}initial_inputs.npy")
)
y_obs = np.asarray(
    np.load(f"f{FUNCTION_NUMBER}initial_outputs.npy")
).ravel()

N, dim = X_obs.shape

print("="*60)
print(f"F{FUNCTION_NUMBER}: GP FITTING AND UCB MAXIMISATION")
print("="*60)
print(f"\nData loaded:")
print(f"  Observations:  {N}")
print(f"  Dimensions:    {dim}")
print(f"  y range:       [{y_obs.min():.6f}, {y_obs.max():.6f}]")
print(f"  y mean:        {y_obs.mean():.6f}")
print(f"  y std:         {y_obs.std():.6f}")

# ── 2. Build bounds dynamically ───────────────────────────────
# Automatically creates correct number of bounds
# for whatever dimension the loaded file has
bounds = [(0.0, 1.0)] * dim
lows   = np.array([b[0] for b in bounds])
highs  = np.array([b[1] for b in bounds])

assert len(bounds) == dim, "Bounds must match dimensionality"
print(f"  Bounds:        (0.0, 1.0) × {dim} dimensions")

# ── 3. Fit GP ─────────────────────────────────────────────────
# Anisotropic RBF: separate length scale per dimension
# This is important for high-D as different dimensions
# may have very different sensitivities
kernel = ConstantKernel(
    1.0,
    constant_value_bounds=(1e-3, 1e3)
) * RBF(
    length_scale=np.ones(dim),
    length_scale_bounds=(0.01, 10.0)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    n_restarts_optimizer=10,
    normalize_y=True
)

print(f"\nFitting GP...")
gp.fit(X_obs, y_obs)
print(f"Fitted kernel: {gp.kernel_}")

# ── 4. Random search for highest UCB ──────────────────────────
# meshgrid is NOT used because:
#
# Dimension | meshgrid points (n=50) | Memory
# ──────────|───────────────────────|──────────────
#     5D    |     312,500,000        |  ~12  GB
#     6D    |  15,625,000,000        |  ~600 GB
#     7D    | 781,250,000,000        |  ~30  TB
#     8D    |         too large      |  impossible
#
# Random sampling with 100,000 points is tractable
# for any dimension and still finds good candidates

print(f"\nSearching for highest UCB...")
print(f"  Random samples: {N_RANDOM:,}")
print(f"  Batch size:     {BATCH_SIZE:,}")

np.random.seed(42)
X_candidates = np.random.uniform(
    low=lows,
    high=highs,
    size=(N_RANDOM, dim)
)

# Predict in batches to avoid memory issues
n_batches = int(np.ceil(N_RANDOM / BATCH_SIZE))
mu_all    = np.zeros(N_RANDOM)
std_all   = np.zeros(N_RANDOM)

for batch in range(n_batches):
    start = batch * BATCH_SIZE
    end   = min(start + BATCH_SIZE, N_RANDOM)
    mu_all[start:end], std_all[start:end] = gp.predict(
        X_candidates[start:end],
        return_std=True
    )

ucb_all      = mu_all + KAPPA * std_all
best_rnd_idx = np.argmax(ucb_all)
best_rnd_pt  = X_candidates[best_rnd_idx]

print(f"\nRandom search best:")
print(f"  UCB:  {ucb_all[best_rnd_idx]:.4f}")
print(f"  Mean: {mu_all[best_rnd_idx]:.4f}")
print(f"  Std:  {std_all[best_rnd_idx]:.4f}")

# ── 5. Refine with scipy optimiser ────────────────────────────
# Polishes the random search result using gradient-based
# local optimisation to find a more precise UCB maximum

def negative_ucb(x):
    x = np.atleast_2d(x)
    mu, std = gp.predict(x, return_std=True)
    return -(mu[0] + KAPPA * std[0])

print(f"\nRefining with scipy L-BFGS-B...")
result = minimize(
    negative_ucb,
    x0=best_rnd_pt,
    method='L-BFGS-B',
    bounds=bounds
)

# Use refined point only if it improves UCB
if -result.fun > ucb_all[best_rnd_idx]:
    next_point       = result.x
    mu_next, std_next = gp.predict(
        next_point.reshape(1, -1),
        return_std=True
    )
    mu_next  = mu_next[0]
    std_next = std_next[0]
    ucb_next = mu_next + KAPPA * std_next
    print(f"  ✓ Refinement improved UCB to {ucb_next:.4f}")
else:
    next_point = best_rnd_pt
    mu_next    = mu_all[best_rnd_idx]
    std_next   = std_all[best_rnd_idx]
    ucb_next   = ucb_all[best_rnd_idx]
    print(f"  ○ Random search result retained")

# ── 6. Current best observation ───────────────────────────────
best_obs_idx = np.argmax(y_obs)
best_obs_val = y_obs[best_obs_idx]
best_obs_loc = X_obs[best_obs_idx]

# ── 7. Print results ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"RESULTS: F{FUNCTION_NUMBER} ({dim}D)")
print(f"{'='*60}")

print(f"\nNext point to sample:")
for d, val in enumerate(next_point):
    print(f"  x{d}: {val:.6f}")

print(f"\nExpected UCB:        {ucb_next:.4f}")
print(f"Expected mean:       {mu_next:.4f}")
print(f"Uncertainty:         {std_next:.4f}")

print(f"\nCurrent best observation:")
print(f"  Value:             {best_obs_val:.6f}")
print(f"  Location:          {best_obs_loc}")

print(f"\nUCB vs current best: {ucb_next - best_obs_val:+.4f}")
if ucb_next > best_obs_val:
    print(f"  ✓ Next point is expected to improve on current best")
else:
    print(f"  ○ Improvement uncertain - high exploration component")

F6: GP FITTING AND UCB MAXIMISATION

Data loaded:
  Observations:  23
  Dimensions:    5
  y range:       [-2.571170, -0.696207]
  y mean:        -1.435343
  y std:         0.461754
  Bounds:        (0.0, 1.0) × 5 dimensions

Fitting GP...
Fitted kernel: 1.34**2 * RBF(length_scale=[0.51, 0.568, 0.829, 0.963, 0.474])

Searching for highest UCB...
  Random samples: 100,000
  Batch size:     10,000

Random search best:
  UCB:  0.9280
  Mean: -0.7031
  Std:  0.4078

Refining with scipy L-BFGS-B...
  ✓ Refinement improved UCB to 0.9735

RESULTS: F6 (5D)

Next point to sample:
  x0: 0.214038
  x1: 0.000000
  x2: 1.000000
  x3: 1.000000
  x4: 0.000000

Expected UCB:        0.9735
Expected mean:       -0.7459
Uncertainty:         0.4298

Current best observation:
  Value:             -0.696207
  Location:          [0.177508 0.387041 0.342993 0.664788 0.059064]

UCB vs current best: +1.6697
  ✓ Next point is expected to improve on current best


In [42]:
# F7 next UCB values
# ── Configuration ─────────────────────────────────────────────
# Change this number to 5, 6, 7, or 8 for F5-F8
FUNCTION_NUMBER = 7
KAPPA           = 4.0
N_RANDOM        = 100000
BATCH_SIZE      = 10000

# ── 1. Load data ──────────────────────────────────────────────
X_obs = np.atleast_2d(
    np.load(f"f{FUNCTION_NUMBER}initial_inputs.npy")
)
y_obs = np.asarray(
    np.load(f"f{FUNCTION_NUMBER}initial_outputs.npy")
).ravel()

N, dim = X_obs.shape

print("="*60)
print(f"F{FUNCTION_NUMBER}: GP FITTING AND UCB MAXIMISATION")
print("="*60)
print(f"\nData loaded:")
print(f"  Observations:  {N}")
print(f"  Dimensions:    {dim}")
print(f"  y range:       [{y_obs.min():.6f}, {y_obs.max():.6f}]")
print(f"  y mean:        {y_obs.mean():.6f}")
print(f"  y std:         {y_obs.std():.6f}")

# ── 2. Build bounds dynamically ───────────────────────────────
# Automatically creates correct number of bounds
# for whatever dimension the loaded file has
bounds = [(0.0, 1.0)] * dim
lows   = np.array([b[0] for b in bounds])
highs  = np.array([b[1] for b in bounds])

assert len(bounds) == dim, "Bounds must match dimensionality"
print(f"  Bounds:        (0.0, 1.0) × {dim} dimensions")

# ── 3. Fit GP ─────────────────────────────────────────────────
# Anisotropic RBF: separate length scale per dimension
# This is important for high-D as different dimensions
# may have very different sensitivities
kernel = ConstantKernel(
    1.0,
    constant_value_bounds=(1e-3, 1e3)
) * RBF(
    length_scale=np.ones(dim),
    length_scale_bounds=(0.01, 10.0)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    n_restarts_optimizer=10,
    normalize_y=True
)

print(f"\nFitting GP...")
gp.fit(X_obs, y_obs)
print(f"Fitted kernel: {gp.kernel_}")

# ── 4. Random search for highest UCB ──────────────────────────
# meshgrid is NOT used because:
#
# Dimension | meshgrid points (n=50) | Memory
# ──────────|───────────────────────|──────────────
#     5D    |     312,500,000        |  ~12  GB
#     6D    |  15,625,000,000        |  ~600 GB
#     7D    | 781,250,000,000        |  ~30  TB
#     8D    |         too large      |  impossible
#
# Random sampling with 100,000 points is tractable
# for any dimension and still finds good candidates

print(f"\nSearching for highest UCB...")
print(f"  Random samples: {N_RANDOM:,}")
print(f"  Batch size:     {BATCH_SIZE:,}")

np.random.seed(42)
X_candidates = np.random.uniform(
    low=lows,
    high=highs,
    size=(N_RANDOM, dim)
)

# Predict in batches to avoid memory issues
n_batches = int(np.ceil(N_RANDOM / BATCH_SIZE))
mu_all    = np.zeros(N_RANDOM)
std_all   = np.zeros(N_RANDOM)

for batch in range(n_batches):
    start = batch * BATCH_SIZE
    end   = min(start + BATCH_SIZE, N_RANDOM)
    mu_all[start:end], std_all[start:end] = gp.predict(
        X_candidates[start:end],
        return_std=True
    )

ucb_all      = mu_all + KAPPA * std_all
best_rnd_idx = np.argmax(ucb_all)
best_rnd_pt  = X_candidates[best_rnd_idx]

print(f"\nRandom search best:")
print(f"  UCB:  {ucb_all[best_rnd_idx]:.4f}")
print(f"  Mean: {mu_all[best_rnd_idx]:.4f}")
print(f"  Std:  {std_all[best_rnd_idx]:.4f}")

# ── 5. Refine with scipy optimiser ────────────────────────────
# Polishes the random search result using gradient-based
# local optimisation to find a more precise UCB maximum

def negative_ucb(x):
    x = np.atleast_2d(x)
    mu, std = gp.predict(x, return_std=True)
    return -(mu[0] + KAPPA * std[0])

print(f"\nRefining with scipy L-BFGS-B...")
result = minimize(
    negative_ucb,
    x0=best_rnd_pt,
    method='L-BFGS-B',
    bounds=bounds
)

# Use refined point only if it improves UCB
if -result.fun > ucb_all[best_rnd_idx]:
    next_point       = result.x
    mu_next, std_next = gp.predict(
        next_point.reshape(1, -1),
        return_std=True
    )
    mu_next  = mu_next[0]
    std_next = std_next[0]
    ucb_next = mu_next + KAPPA * std_next
    print(f"  ✓ Refinement improved UCB to {ucb_next:.4f}")
else:
    next_point = best_rnd_pt
    mu_next    = mu_all[best_rnd_idx]
    std_next   = std_all[best_rnd_idx]
    ucb_next   = ucb_all[best_rnd_idx]
    print(f"  ○ Random search result retained")

# ── 6. Current best observation ───────────────────────────────
best_obs_idx = np.argmax(y_obs)
best_obs_val = y_obs[best_obs_idx]
best_obs_loc = X_obs[best_obs_idx]

# ── 7. Print results ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"RESULTS: F{FUNCTION_NUMBER} ({dim}D)")
print(f"{'='*60}")

print(f"\nNext point to sample:")
for d, val in enumerate(next_point):
    print(f"  x{d}: {val:.6f}")

print(f"\nExpected UCB:        {ucb_next:.4f}")
print(f"Expected mean:       {mu_next:.4f}")
print(f"Uncertainty:         {std_next:.4f}")

print(f"\nCurrent best observation:")
print(f"  Value:             {best_obs_val:.6f}")
print(f"  Location:          {best_obs_loc}")

print(f"\nUCB vs current best: {ucb_next - best_obs_val:+.4f}")
if ucb_next > best_obs_val:
    print(f"  ✓ Next point is expected to improve on current best")
else:
    print(f"  ○ Improvement uncertain - high exploration component")

F7: GP FITTING AND UCB MAXIMISATION

Data loaded:
  Observations:  33
  Dimensions:    6
  y range:       [0.002701, 1.364968]
  y mean:        0.232283
  y std:         0.300768
  Bounds:        (0.0, 1.0) × 6 dimensions

Fitting GP...
Fitted kernel: 1.06**2 * RBF(length_scale=[0.502, 4.27, 10, 0.133, 0.141, 1])

Searching for highest UCB...
  Random samples: 100,000
  Batch size:     10,000

Random search best:
  UCB:  1.9674
  Mean: 0.9711
  Std:  0.2491

Refining with scipy L-BFGS-B...
  ✓ Refinement improved UCB to 1.9737

RESULTS: F7 (6D)

Next point to sample:
  x0: 0.164844
  x1: 0.000000
  x2: 1.000000
  x3: 0.325292
  x4: 0.451013
  x5: 1.000000

Expected UCB:        1.9737
Expected mean:       1.0394
Uncertainty:         0.2336

Current best observation:
  Value:             1.364968
  Location:          [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]

UCB vs current best: +0.6088
  ✓ Next point is expected to improve on current best


In [43]:
# F8 next UCB values
# ── Configuration ─────────────────────────────────────────────
# Change this number to 5, 6, 7, or 8 for F5-F8
FUNCTION_NUMBER = 8
KAPPA           = 4.0
N_RANDOM        = 100000
BATCH_SIZE      = 10000

# ── 1. Load data ──────────────────────────────────────────────
X_obs = np.atleast_2d(
    np.load(f"f{FUNCTION_NUMBER}initial_inputs.npy")
)
y_obs = np.asarray(
    np.load(f"f{FUNCTION_NUMBER}initial_outputs.npy")
).ravel()

N, dim = X_obs.shape

print("="*60)
print(f"F{FUNCTION_NUMBER}: GP FITTING AND UCB MAXIMISATION")
print("="*60)
print(f"\nData loaded:")
print(f"  Observations:  {N}")
print(f"  Dimensions:    {dim}")
print(f"  y range:       [{y_obs.min():.6f}, {y_obs.max():.6f}]")
print(f"  y mean:        {y_obs.mean():.6f}")
print(f"  y std:         {y_obs.std():.6f}")

# ── 2. Build bounds dynamically ───────────────────────────────
# Automatically creates correct number of bounds
# for whatever dimension the loaded file has
bounds = [(0.0, 1.0)] * dim
lows   = np.array([b[0] for b in bounds])
highs  = np.array([b[1] for b in bounds])

assert len(bounds) == dim, "Bounds must match dimensionality"
print(f"  Bounds:        (0.0, 1.0) × {dim} dimensions")

# ── 3. Fit GP ─────────────────────────────────────────────────
# Anisotropic RBF: separate length scale per dimension
# This is important for high-D as different dimensions
# may have very different sensitivities
kernel = ConstantKernel(
    1.0,
    constant_value_bounds=(1e-3, 1e3)
) * RBF(
    length_scale=np.ones(dim),
    length_scale_bounds=(0.01, 10.0)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,
    n_restarts_optimizer=10,
    normalize_y=True
)

print(f"\nFitting GP...")
gp.fit(X_obs, y_obs)
print(f"Fitted kernel: {gp.kernel_}")

# ── 4. Random search for highest UCB ──────────────────────────
# meshgrid is NOT used because:
#
# Dimension | meshgrid points (n=50) | Memory
# ──────────|───────────────────────|──────────────
#     5D    |     312,500,000        |  ~12  GB
#     6D    |  15,625,000,000        |  ~600 GB
#     7D    | 781,250,000,000        |  ~30  TB
#     8D    |         too large      |  impossible
#
# Random sampling with 100,000 points is tractable
# for any dimension and still finds good candidates

print(f"\nSearching for highest UCB...")
print(f"  Random samples: {N_RANDOM:,}")
print(f"  Batch size:     {BATCH_SIZE:,}")

np.random.seed(42)
X_candidates = np.random.uniform(
    low=lows,
    high=highs,
    size=(N_RANDOM, dim)
)

# Predict in batches to avoid memory issues
n_batches = int(np.ceil(N_RANDOM / BATCH_SIZE))
mu_all    = np.zeros(N_RANDOM)
std_all   = np.zeros(N_RANDOM)

for batch in range(n_batches):
    start = batch * BATCH_SIZE
    end   = min(start + BATCH_SIZE, N_RANDOM)
    mu_all[start:end], std_all[start:end] = gp.predict(
        X_candidates[start:end],
        return_std=True
    )

ucb_all      = mu_all + KAPPA * std_all
best_rnd_idx = np.argmax(ucb_all)
best_rnd_pt  = X_candidates[best_rnd_idx]

print(f"\nRandom search best:")
print(f"  UCB:  {ucb_all[best_rnd_idx]:.4f}")
print(f"  Mean: {mu_all[best_rnd_idx]:.4f}")
print(f"  Std:  {std_all[best_rnd_idx]:.4f}")

# ── 5. Refine with scipy optimiser ────────────────────────────
# Polishes the random search result using gradient-based
# local optimisation to find a more precise UCB maximum

def negative_ucb(x):
    x = np.atleast_2d(x)
    mu, std = gp.predict(x, return_std=True)
    return -(mu[0] + KAPPA * std[0])

print(f"\nRefining with scipy L-BFGS-B...")
result = minimize(
    negative_ucb,
    x0=best_rnd_pt,
    method='L-BFGS-B',
    bounds=bounds
)

# Use refined point only if it improves UCB
if -result.fun > ucb_all[best_rnd_idx]:
    next_point       = result.x
    mu_next, std_next = gp.predict(
        next_point.reshape(1, -1),
        return_std=True
    )
    mu_next  = mu_next[0]
    std_next = std_next[0]
    ucb_next = mu_next + KAPPA * std_next
    print(f"  ✓ Refinement improved UCB to {ucb_next:.4f}")
else:
    next_point = best_rnd_pt
    mu_next    = mu_all[best_rnd_idx]
    std_next   = std_all[best_rnd_idx]
    ucb_next   = ucb_all[best_rnd_idx]
    print(f"  ○ Random search result retained")

# ── 6. Current best observation ───────────────────────────────
best_obs_idx = np.argmax(y_obs)
best_obs_val = y_obs[best_obs_idx]
best_obs_loc = X_obs[best_obs_idx]

# ── 7. Print results ──────────────────────────────────────────
print(f"\n{'='*60}")
print(f"RESULTS: F{FUNCTION_NUMBER} ({dim}D)")
print(f"{'='*60}")

print(f"\nNext point to sample:")
for d, val in enumerate(next_point):
    print(f"  x{d}: {val:.6f}")

print(f"\nExpected UCB:        {ucb_next:.4f}")
print(f"Expected mean:       {mu_next:.4f}")
print(f"Uncertainty:         {std_next:.4f}")

print(f"\nCurrent best observation:")
print(f"  Value:             {best_obs_val:.6f}")
print(f"  Location:          {best_obs_loc}")

print(f"\nUCB vs current best: {ucb_next - best_obs_val:+.4f}")
if ucb_next > best_obs_val:
    print(f"  ✓ Next point is expected to improve on current best")
else:
    print(f"  ○ Improvement uncertain - high exploration component")

F8: GP FITTING AND UCB MAXIMISATION

Data loaded:
  Observations:  43
  Dimensions:    8
  y range:       [4.039975, 9.953954]
  y mean:        7.704763
  y std:         1.218771
  Bounds:        (0.0, 1.0) × 8 dimensions

Fitting GP...
Fitted kernel: 1.55**2 * RBF(length_scale=[1.05, 1.44, 0.662, 3.02, 5.67, 10, 1.07, 10])

Searching for highest UCB...
  Random samples: 100,000
  Batch size:     10,000

Random search best:
  UCB:  11.3611
  Mean: 9.4754
  Std:  0.4714

Refining with scipy L-BFGS-B...
  ✓ Refinement improved UCB to 11.7180

RESULTS: F8 (8D)

Next point to sample:
  x0: 0.000000
  x1: 1.000000
  x2: 0.000000
  x3: 1.000000
  x4: 1.000000
  x5: 1.000000
  x6: 0.000000
  x7: 0.000000

Expected UCB:        11.7180
Expected mean:       9.4205
Uncertainty:         0.5744

Current best observation:
  Value:             9.953954
  Location:          [0.130265 0.046283 0.172839 0.194561 0.988438 0.432    0.222147 0.761423]

UCB vs current best: +1.7641
  ✓ Next point is expecte

In [ ]:
# F3 - 2s sliced plot
plot_2d_slices(gp, X_obs, y_obs, bounds=[(0,1), (0,1), (0,1)], kappa=3.0)

In [16]:
def comprehensive_gp_assessment(gp, X_obs, y_obs, bounds, kappa=3.0, n_random_samples=5000):
    """
    Complete non-visual assessment of GP optimization progress.
    Works for any dimensionality.
    
    Args:
        gp: Fitted GaussianProcessRegressor
        X_obs: Observed inputs (N, D)
        y_obs: Observed outputs (N,)
        bounds: List of (min, max) tuples for each dimension
        kappa: UCB exploration parameter
        n_random_samples: Number of random points to sample for global assessment
    
    Returns:
        dict: Comprehensive assessment results
    """
    dim = len(bounds)
    n_obs = len(y_obs)
    
    print("\n" + "="*80)
    print(f"COMPREHENSIVE GP OPTIMIZATION ASSESSMENT ({dim}D)")
    print("="*80)
    
    # ===================================================================
    # 1. BASIC STATISTICS
    # ===================================================================
    print("\n" + "─"*80)
    print("1. BASIC STATISTICS")
    print("─"*80)
    
    best_idx = np.argmax(y_obs)
    best_x = X_obs[best_idx]
    best_y = y_obs[best_idx]
    
    print(f"Dimensionality:          {dim}D")
    print(f"Total observations:      {n_obs}")
    print(f"Samples per dimension:   {n_obs/dim:.2f}")
    print(f"Sampling density:        {n_obs / (dim * 10):.1f}x baseline (10 per dim)")
    
    if n_obs < 5 * dim:
        print("⚠️  WARNING: Very sparse sampling for this dimensionality!")
    elif n_obs < 10 * dim:
        print("⚠️  NOTICE: Moderate sampling - consider more observations")
    else:
        print("✓  Good sampling density")
    
    # ===================================================================
    # 2. CURRENT BEST POINT ANALYSIS
    # ===================================================================
    print("\n" + "─"*80)
    print("2. CURRENT BEST POINT")
    print("─"*80)
    
    print(f"Location:                {best_x}")
    print(f"Value:                   {best_y:.6f}")
    
    # Uncertainty at best point
    _, std_at_best = gp.predict(best_x.reshape(1, -1), return_std=True)
    std_at_best = std_at_best[0]
    
    print(f"Uncertainty (σ):         {std_at_best:.6f}")
    
    # Confidence assessment
    if std_at_best < 0.01:
        confidence = "VERY HIGH"
        symbol = "✓✓"
    elif std_at_best < 0.05:
        confidence = "HIGH"
        symbol = "✓"
    elif std_at_best < 0.1:
        confidence = "MODERATE"
        symbol = "○"
    else:
        confidence = "LOW"
        symbol = "⚠️"
    
    print(f"Confidence level:        {confidence} {symbol}")
    
    # Check if on boundary
    lows = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])
    
    distances_to_lower = best_x - lows
    distances_to_upper = highs - best_x
    min_distances = np.minimum(distances_to_lower, distances_to_upper)
    
    boundary_threshold = 0.05  # 5% of range
    on_boundary_dims = np.where(min_distances < boundary_threshold)[0]
    
    if len(on_boundary_dims) > 0:
        print(f"⚠️  ON BOUNDARY: Dimensions {on_boundary_dims.tolist()}")
        print(f"   Consider expanding search space in these dimensions")
    else:
        print(f"✓  Not on boundary - good interior solution")
    
    # ===================================================================
    # 3. PERFORMANCE DISTRIBUTION
    # ===================================================================
    print("\n" + "─"*80)
    print("3. PERFORMANCE DISTRIBUTION")
    print("─"*80)
    
    y_min = y_obs.min()
    y_max = y_obs.max()
    y_mean = y_obs.mean()
    y_std = y_obs.std()
    y_range = y_max - y_min
    
    print(f"Min value:               {y_min:.6f}")
    print(f"Max value:               {y_max:.6f}")
    print(f"Mean value:              {y_mean:.6f}")
    print(f"Std deviation:           {y_std:.6f}")
    print(f"Range:                   {y_range:.6f}")
    
    # Check for flat function
    if y_range < 0.01 * abs(y_mean):
        print("⚠️  WARNING: Very flat function - may need rescaling or different objective")
    elif y_range < 0.1 * abs(y_mean):
        print("⚠️  NOTICE: Limited variation in objective")
    else:
        print("✓  Good dynamic range")
    
    # Percentiles
    percentiles = [10, 25, 50, 75, 90]
    perc_values = np.percentile(y_obs, percentiles)
    print(f"\nPercentiles:")
    for p, v in zip(percentiles, perc_values):
        print(f"  {p}th percentile:        {v:.6f}")
    
    # ===================================================================
    # 4. KERNEL ANALYSIS
    # ===================================================================
    print("\n" + "─"*80)
    print("4. KERNEL PARAMETERS")
    print("─"*80)
    
    print(f"Kernel:                  {gp.kernel_}")
    
    # Extract length scales if available
    if hasattr(gp.kernel_, 'k2') and hasattr(gp.kernel_.k2, 'length_scale'):
        # For ConstantKernel * RBF structure
        length_scales = np.atleast_1d(gp.kernel_.k2.length_scale)
    elif hasattr(gp.kernel_, 'length_scale'):
        length_scales = np.atleast_1d(gp.kernel_.length_scale)
    else:
        length_scales = None
    
    if length_scales is not None:
        print(f"\nLength scales:")
        if len(length_scales) == 1:
            # Isotropic kernel
            print(f"  Isotropic:             {length_scales[0]:.4f}")
            if length_scales[0] < 0.05:
                print(f"  ⚠️  Very short - function very wiggly, may not generalize well")
            elif length_scales[0] > 5.0:
                print(f"  ⚠️  Very long - may be over-smoothing")
            else:
                print(f"  ✓  Reasonable length scale")
        else:
            # Anisotropic kernel
            print(f"  Anisotropic (per dimension):")
            for i, ls in enumerate(length_scales):
                status = "✓" if 0.05 < ls < 5.0 else "⚠️"
                print(f"    x{i}: {ls:.4f} {status}")
            
            print(f"\n  Mean length scale:     {length_scales.mean():.4f}")
            print(f"  Min/Max:               {length_scales.min():.4f} / {length_scales.max():.4f}")
            print(f"  Ratio (max/min):       {length_scales.max() / length_scales.min():.2f}")
            
            # Identify most/least sensitive dimensions
            sorted_dims = np.argsort(length_scales)
            print(f"\n  Most sensitive dims:   {sorted_dims[:3].tolist()} (short length scale)")
            print(f"  Least sensitive dims:  {sorted_dims[-3:].tolist()} (long length scale)")
    
    # ===================================================================
    # 5. UNCERTAINTY ANALYSIS
    # ===================================================================
    print("\n" + "─"*80)
    print("5. UNCERTAINTY ANALYSIS")
    print("─"*80)
    
    # Uncertainty at observations
    mu_obs, std_obs = gp.predict(X_obs, return_std=True)
    
    print(f"At observed points:")
    print(f"  Mean uncertainty:      {std_obs.mean():.6f}")
    print(f"  Min uncertainty:       {std_obs.min():.6f}")
    print(f"  Max uncertainty:       {std_obs.max():.6f}")
    
    # Sample random unexplored points
    X_random = np.random.uniform(
        low=lows,
        high=highs,
        size=(n_random_samples, dim)
    )
    
    mu_random, std_random = gp.predict(X_random, return_std=True)
    
    print(f"\nAt random unexplored points:")
    print(f"  Mean uncertainty:      {std_random.mean():.6f}")
    print(f"  Min uncertainty:       {std_random.min():.6f}")
    print(f"  Max uncertainty:       {std_random.max():.6f}")
    
    # Uncertainty reduction ratio
    uncertainty_ratio = std_obs.mean() / std_random.mean()
    print(f"\nUncertainty reduction:   {uncertainty_ratio:.2f}x")
    print(f"  (How much more certain we are at observations vs unexplored)")
    
    if uncertainty_ratio > 0.8:
        print(f"  ⚠️  Limited uncertainty reduction - kernel may not be learning well")
    elif uncertainty_ratio > 0.5:
        print(f"  ○  Moderate learning")
    else:
        print(f"  ✓  Good learning - observations reducing uncertainty effectively")
    
    # ===================================================================
    # 6. UCB ACQUISITION ANALYSIS
    # ===================================================================
    print("\n" + "─"*80)
    print("6. UCB ACQUISITION FUNCTION")
    print("─"*80)
    
    ucb_obs = mu_obs + kappa * std_obs
    ucb_random = mu_random + kappa * std_random
    
    print(f"UCB at observed points:")
    print(f"  Mean:                  {ucb_obs.mean():.6f}")
    print(f"  Max:                   {ucb_obs.max():.6f}")
    
    print(f"\nUCB at random points:")
    print(f"  Mean:                  {ucb_random.mean():.6f}")
    print(f"  Max:                   {ucb_random.max():.6f}")
    
    # Find best UCB point
    best_ucb_idx = np.argmax(ucb_random)
    best_ucb_point = X_random[best_ucb_idx]
    best_ucb_value = ucb_random[best_ucb_idx]
    
    print(f"\nNext suggested point:")
    print(f"  Location:              {best_ucb_point}")
    print(f"  UCB value:             {best_ucb_value:.6f}")
    print(f"  Expected mean:         {mu_random[best_ucb_idx]:.6f}")
    print(f"  Uncertainty:           {std_random[best_ucb_idx]:.6f}")
    
    # Compare to current best
    ucb_improvement_potential = best_ucb_value - best_y
    
    print(f"\nImprovement potential:")
    print(f"  UCB gap:               {ucb_improvement_potential:.6f}")
    
    if ucb_improvement_potential > 2 * std_at_best:
        print(f"  🎯 HIGH potential - unexplored regions may be significantly better")
        recommendation = "EXPLORE"
    elif ucb_improvement_potential > std_at_best:
        print(f"  ○  MODERATE potential - worth exploring")
        recommendation = "EXPLORE"
    else:
        print(f"  ✓  LOW potential - current best likely near optimal")
        recommendation = "REFINE or STOP"
    
    # ===================================================================
    # 7. CONVERGENCE INDICATORS
    # ===================================================================
    print("\n" + "─"*80)
    print("7. CONVERGENCE ANALYSIS")
    print("─"*80)
    
    # Best value progression
    cumulative_best = np.maximum.accumulate(y_obs)
    
    if n_obs >= 10:
        # Recent improvement
        recent_window = min(10, n_obs // 2)
        recent_improvement = cumulative_best[-1] - cumulative_best[-recent_window]
        
        print(f"Improvement over last {recent_window} samples:")
        print(f"  Absolute:              {recent_improvement:.6f}")
        print(f"  Relative to range:     {100 * recent_improvement / y_range:.2f}%")
        
        # Convergence threshold: < 1% of range
        convergence_threshold = 0.01 * y_range
        
        if abs(recent_improvement) < convergence_threshold:
            print(f"  ✓  CONVERGING - little recent improvement")
            converging = True
        else:
            print(f"  ○  STILL IMPROVING - not yet converged")
            converging = False
    else:
        print(f"  ⚠️  Too few samples to assess convergence")
        recent_improvement = None
        converging = False
    
    # Stagnation detection
    if n_obs >= 5:
        last_5_best = cumulative_best[-5:]
        if len(np.unique(last_5_best)) == 1:
            print(f"  ⚠️  STAGNATED - no improvement in last 5 samples")
            stagnated = True
        else:
            stagnated = False
    else:
        stagnated = False
    
    # ===================================================================
    # 8. EXPLORATION COVERAGE
    # ===================================================================
    print("\n" + "─"*80)
    print("8. EXPLORATION COVERAGE")
    print("─"*80)
    
    # Pairwise distances between observations
    from scipy.spatial.distance import pdist, squareform
    
    # Normalize to [0,1] for each dimension
    X_normalized = (X_obs - lows) / (highs - lows)
    distances = pdist(X_normalized)
    
    print(f"Pairwise distances (normalized):")
    print(f"  Mean:                  {distances.mean():.4f}")
    print(f"  Min:                   {distances.min():.4f}")
    print(f"  Max:                   {distances.max():.4f}")
    
    # Check for clustering
    if distances.min() < 0.05:
        print(f"  ⚠️  Some points very close - possible over-exploitation")
    
    # Coverage in each dimension
    print(f"\nCoverage per dimension:")
    for i in range(dim):
        dim_range = X_obs[:, i].max() - X_obs[:, i].min()
        dim_span = bounds[i][1] - bounds[i][0]
        coverage = 100 * dim_range / dim_span
        
        status = "✓" if coverage > 70 else "⚠️" if coverage > 40 else "✗"
        print(f"  x{i}: {coverage:.1f}% {status}")
    
    # ===================================================================
    # 9. MODEL QUALITY ASSESSMENT
    # ===================================================================
    print("\n" + "─"*80)
    print("9. MODEL QUALITY (Cross-Validation)")
    print("─"*80)
    
    # Leave-one-out cross-validation
    residuals = []
    standardized_residuals = []
    
    for i in range(min(n_obs, 20)):  # Limit to 20 for speed
        # Leave one out
        X_train = np.delete(X_obs, i, axis=0)
        y_train = np.delete(y_obs, i)
        X_test = X_obs[i:i+1]
        y_test = y_obs[i]
        
        # Fit and predict
        gp_cv = GaussianProcessRegressor(
            kernel=gp.kernel_,
            alpha=gp.alpha,
            normalize_y=gp.normalize_y
        )
        gp_cv.fit(X_train, y_train)
        mu_pred, std_pred = gp_cv.predict(X_test, return_std=True)
        
        residual = y_test - mu_pred[0]
        residuals.append(residual)
        standardized_residuals.append(residual / std_pred[0])
    
    residuals = np.array(residuals)
    standardized_residuals = np.array(standardized_residuals)
    
    print(f"Leave-one-out validation ({len(residuals)} samples):")
    print(f"  Mean abs error:        {np.abs(residuals).mean():.6f}")
    print(f"  RMSE:                  {np.sqrt((residuals**2).mean()):.6f}")
    print(f"  Mean std residual:     {np.abs(standardized_residuals).mean():.4f}")
    
    # Good GP should have ~68% within 1 sigma
    within_1sigma = np.sum(np.abs(standardized_residuals) < 1) / len(standardized_residuals)
    print(f"  Within 1σ:             {100*within_1sigma:.1f}% (expect ~68%)")
    
    if within_1sigma < 0.5:
        print(f"  ⚠️  Uncertainty estimates may be inaccurate")
    elif within_1sigma > 0.8:
        print(f"  ⚠️  May be over-confident (too narrow uncertainties)")
    else:
        print(f"  ✓  Uncertainty calibration looks good")
    
    # ===================================================================
    # 10. FINAL RECOMMENDATIONS
    # ===================================================================
    print("\n" + "="*80)
    print("10. RECOMMENDATIONS")
    print("="*80)
    
    recommendations = []
    
    # Based on sampling density
    if n_obs < 5 * dim:
        recommendations.append(
            f"⚠️  CRITICAL: Very sparse sampling ({n_obs/dim:.1f} per dim). "
            f"Collect at least {5*dim - n_obs} more samples."
        )
    elif n_obs < 10 * dim:
        recommendations.append(
            f"○  Consider {10*dim - n_obs} more samples for better coverage ({n_obs/dim:.1f} per dim currently)."
        )
    
    # Based on uncertainty at best
    if std_at_best > 0.1:
        recommendations.append(
            f"⚠️  High uncertainty at best point (σ={std_at_best:.4f}). "
            f"Sample nearby to confirm optimum."
        )
    
    # Based on UCB potential
    if ucb_improvement_potential > 2 * std_at_best:
        recommendations.append(
            f"🎯 HIGH UCB potential detected. Next sample: {best_ucb_point}"
        )
    elif ucb_improvement_potential > std_at_best:
        recommendations.append(
            f"○  Moderate exploration potential remains. Suggest {3-5} more samples."
        )
    
    # Based on boundary
    if len(on_boundary_dims) > 0:
        recommendations.append(
            f"⚠️  Optimum on boundary (dims {on_boundary_dims.tolist()}). "
            f"Consider expanding search space."
        )
    
    # Based on convergence
    if converging and std_at_best < 0.05:
        recommendations.append(
            f"✓  Optimization appears converged. Current best is likely optimal."
        )
    elif stagnated:
        recommendations.append(
            f"⚠️  Stagnated without convergence. Try: (1) Increase exploration (higher κ), "
            f"(2) Check kernel, or (3) Expand search space."
        )
    
    # Based on kernel
    if length_scales is not None:
        if np.any(length_scales < 0.05):
            recommendations.append(
                f"⚠️  Very short length scales detected. Function may be too wiggly - "
                f"consider smoother kernel or check for noise."
            )
        if np.any(length_scales > 5.0):
            recommendations.append(
                f"⚠️  Very long length scales detected. May be over-smoothing - "
                f"consider more flexible kernel."
            )
    
    if len(recommendations) == 0:
        recommendations.append("✓  Optimization progressing well. Continue as planned.")
    
    for i, rec in enumerate(recommendations, 1):
        print(f"\n{i}. {rec}")
    
    print("\n" + "="*80)
    
    # ===================================================================
    # RETURN SUMMARY
    # ===================================================================
    
    return {
        'best_x': best_x,
        'best_y': best_y,
        'uncertainty_at_best': std_at_best,
        'confidence': confidence,
        'on_boundary': len(on_boundary_dims) > 0,
        'boundary_dims': on_boundary_dims.tolist(),
        'next_ucb_point': best_ucb_point,
        'next_ucb_value': best_ucb_value,
        'ucb_improvement_potential': ucb_improvement_potential,
        'recommendation': recommendation,
        'converging': converging,
        'stagnated': stagnated,
        'n_obs': n_obs,
        'dim': dim,
        'length_scales': length_scales.tolist() if length_scales is not None else None,
        'recommendations': recommendations
    }

In [36]:
# USAGE - comprehensive GP assessment for F4
# ===================================================================

# Load data
X4_obs = np.load("f4initial_inputs.npy")
y4_obs = np.load("f4initial_outputs.npy")
bounds = [(0.0, 1.0), (0.0, 1.0), (0.0, 1.0), (0.0, 1.0)]
kappa = 10.0

X_obs = np.atleast_2d(X4_obs)
y_obs = np.asarray(y4_obs).ravel()
N, dim = X_obs.shape
assert len(bounds) == dim, "Bounds must match dimensionality"

kernel = ConstantKernel(1.0) * RBF(length_scale=1.0, length_scale_bounds=(0.01, 10.0))
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-6,  # Reduced noise assumption
    n_restarts_optimizer=10,  # Better kernel optimization
    normalize_y=True
)

gp.fit(X_obs, y_obs)

# For your 3D, 4D, or 8D problem:
results = comprehensive_gp_assessment(
    gp=gp,
    X_obs=X_obs,
    y_obs=y_obs,
    bounds=bounds,
    kappa=3.0,
    n_random_samples=5000
)

# Access results programmatically
print(f"\nBest point: {results['best_x']}")
print(f"Recommendation: {results['recommendation']}")
print(f"Should continue? {not results['converging']}")
# ```

# ---

# ## **Key Metrics to Monitor (Summary)**

# Here's what to watch for in the non-visual assessment:

# ### **🔴 RED FLAGS (Stop and Fix)**

# 1. **Very sparse sampling**: < 5 samples per dimension
# 2. **Flat function**: Range < 1% of mean value
# 3. **Boundary optimum**: Best point within 5% of bounds
# 4. **Poor calibration**: < 50% or > 80% of residuals within 1σ
# 5. **Extreme length scales**: < 0.05 or > 5.0
# 6. **High uncertainty at best**: σ > 0.1 at optimum

# ### **🟡 WARNINGS (Proceed with Caution)**

# 1. **Moderate sampling**: 5-10 samples per dimension
# 2. **Stagnation**: No improvement in last 5 samples
# 3. **Limited uncertainty reduction**: Ratio > 0.5
# 4. **Moderate boundary proximity**: Within 10% of bounds
# 5. **High UCB potential**: > 2σ improvement possible

# ### **🟢 GOOD SIGNS (Continue Optimizing)**

# 1. **Dense sampling**: > 10 samples per dimension
# 2. **Low uncertainty at best**: σ < 0.05
# 3. **Interior optimum**: Not on boundary
# 4. **Converging**: Little improvement in recent samples
# 5. **Good calibration**: ~68% residuals within 1σ
# 6. **Low UCB potential**: < 1σ improvement expected

# ---

# ## **Quick Decision Tree**
# # ```
# # Is uncertainty at best < 0.05?
# # ├─ NO → Sample more around best point
# # └─ YES → Is UCB improvement potential > 2σ?
# #     ├─ YES → Explore high-UCB region
# #     └─ NO → Is best on boundary?
# #         ├─ YES → Expand search space
# #         └─ NO → Is converging?
# #             ├─ YES → DONE! ✓
# #             └─ NO → Continue with 5-10 more samples


COMPREHENSIVE GP OPTIMIZATION ASSESSMENT (4D)

────────────────────────────────────────────────────────────────────────────────
1. BASIC STATISTICS
────────────────────────────────────────────────────────────────────────────────
Dimensionality:          4D
Total observations:      33
Samples per dimension:   8.25
Sampling density:        0.8x baseline (10 per dim)
⚠️  NOTICE: Moderate sampling - consider more observations

────────────────────────────────────────────────────────────────────────────────
2. CURRENT BEST POINT
────────────────────────────────────────────────────────────────────────────────
Location:                [0.57776561 0.42877174 0.42582587 0.24900741]
Value:                   -4.025542
Uncertainty (σ):         0.006699
Confidence level:        VERY HIGH ✓✓
✓  Not on boundary - good interior solution

────────────────────────────────────────────────────────────────────────────────
3. PERFORMANCE DISTRIBUTION
────────────────────────────────────────────────────────

In [22]:
def diagnose_kernel(gp, X_obs, y_obs):
    """
    Comprehensive kernel diagnosis.
    """
    print("="*70)
    print("KERNEL DIAGNOSIS")
    print("="*70)
    
    # 1. Print kernel structure
    print(f"\n1. KERNEL STRUCTURE:")
    print(f"   {gp.kernel_}")
    
    # 2. Extract parameters
    kernel = gp.kernel_
    
    # Check for common kernel structures
    if hasattr(kernel, 'k1') and hasattr(kernel, 'k2'):
        # Product kernel (e.g., ConstantKernel * RBF)
        print(f"\n   Type: Product kernel")
        if hasattr(kernel.k1, 'constant_value'):
            print(f"   - Constant scale: {kernel.k1.constant_value:.4f}")
        if hasattr(kernel.k2, 'length_scale'):
            ls = np.atleast_1d(kernel.k2.length_scale)
            print(f"   - Length scale(s): {ls}")
    elif hasattr(kernel, 'length_scale'):
        # Simple RBF
        ls = np.atleast_1d(kernel.length_scale)
        print(f"\n   Type: Simple RBF")
        print(f"   - Length scale(s): {ls}")
    
    # 3. Analyze length scales
    if hasattr(kernel, 'k2') and hasattr(kernel.k2, 'length_scale'):
        length_scales = np.atleast_1d(kernel.k2.length_scale)
    elif hasattr(kernel, 'length_scale'):
        length_scales = np.atleast_1d(kernel.length_scale)
    else:
        print("\n   ⚠️  Cannot extract length scales")
        return None
    
    print(f"\n2. LENGTH SCALE ANALYSIS:")
    dim = X_obs.shape[1]
    
    if len(length_scales) == 1:
        # Isotropic kernel
        ls = length_scales[0]
        print(f"   Isotropic: {ls:.4f}")
        
        if ls < 0.05:
            print(f"   ❌ TOO SHORT - Function too wiggly, won't generalize")
            severity = "CRITICAL"
        elif ls < 0.1:
            print(f"   ⚠️  SHORT - May be overfitting to noise")
            severity = "WARNING"
        elif ls > 5.0:
            print(f"   ❌ TOO LONG - Over-smoothing, missing structure")
            severity = "CRITICAL"
        elif ls > 2.0:
            print(f"   ⚠️  LONG - May be under-fitting")
            severity = "WARNING"
        else:
            print(f"   ✓  GOOD - Reasonable length scale")
            severity = "OK"
    else:
        # Anisotropic kernel
        print(f"   Anisotropic (per dimension):")
        issues = []
        
        for i, ls in enumerate(length_scales):
            if ls < 0.05:
                status = "❌ TOO SHORT"
                issues.append(i)
            elif ls < 0.1:
                status = "⚠️  SHORT"
            elif ls > 5.0:
                status = "❌ TOO LONG"
                issues.append(i)
            elif ls > 2.0:
                status = "⚠️  LONG"
            else:
                status = "✓  GOOD"
            
            print(f"     x{i}: {ls:.4f} {status}")
        
        if issues:
            print(f"\n   ⚠️  Problem dimensions: {issues}")
            severity = "WARNING"
        else:
            print(f"\n   ✓  All length scales reasonable")
            severity = "OK"
        
        # Check for extreme anisotropy
        ratio = length_scales.max() / length_scales.min()
        print(f"\n   Length scale ratio (max/min): {ratio:.2f}")
        if ratio > 100:
            print(f"   ❌ EXTREME anisotropy - some dims ignored")
        elif ratio > 10:
            print(f"   ⚠️  High anisotropy")
        else:
            print(f"   ✓  Moderate anisotropy")
    
    # 4. Check noise level (alpha parameter)
    print(f"\n3. NOISE LEVEL:")
    print(f"   Alpha: {gp.alpha}")
    
    if gp.alpha > 0.1:
        print(f"   ⚠️  High noise assumption - may be under-fitting")
    elif gp.alpha < 1e-8:
        print(f"   ⚠️  Very low noise - may cause numerical issues")
    else:
        print(f"   ✓  Reasonable noise level")
    
    # 5. Model fit quality
    print(f"\n4. MODEL FIT QUALITY:")
    
    # Training predictions
    mu_train, std_train = gp.predict(X_obs, return_std=True)
    residuals = y_obs - mu_train
    
    print(f"   Training RMSE: {np.sqrt(np.mean(residuals**2)):.6f}")
    print(f"   Training MAE:  {np.mean(np.abs(residuals)):.6f}")
    print(f"   Mean std dev:  {std_train.mean():.6f}")
    
    # Check for overfitting/underfitting
    relative_error = np.abs(residuals).mean() / np.abs(y_obs).std()
    
    if relative_error < 0.01:
        print(f"   ⚠️  Possible overfitting (too good fit)")
    elif relative_error > 0.5:
        print(f"   ⚠️  Possible underfitting (poor fit)")
    else:
        print(f"   ✓  Reasonable fit quality")
    
    # 6. Kernel hyperparameter bounds
    print(f"\n5. HYPERPARAMETER BOUNDS:")
    if hasattr(kernel, 'k2') and hasattr(kernel.k2, 'length_scale_bounds'):
        bounds = kernel.k2.length_scale_bounds
        print(f"   Length scale bounds: {bounds}")
        
        # Check if at bounds
        if len(length_scales) == 1:
            if np.isclose(length_scales[0], bounds[0], rtol=0.1):
                print(f"   ⚠️  Length scale at LOWER bound - may need wider bounds")
            elif np.isclose(length_scales[0], bounds[1], rtol=0.1):
                print(f"   ⚠️  Length scale at UPPER bound - may need wider bounds")
            else:
                print(f"   ✓  Not at bounds")
        else:
            at_lower = np.sum(np.isclose(length_scales, bounds[0], rtol=0.1))
            at_upper = np.sum(np.isclose(length_scales, bounds[1], rtol=0.1))
            if at_lower > 0:
                print(f"   ⚠️  {at_lower} dimensions at LOWER bound")
            if at_upper > 0:
                print(f"   ⚠️  {at_upper} dimensions at UPPER bound")
            if at_lower == 0 and at_upper == 0:
                print(f"   ✓  Not at bounds")
    
    # 7. Summary recommendation
    print(f"\n6. OVERALL ASSESSMENT:")
    
    if severity == "CRITICAL":
        print(f"   ❌ CRITICAL ISSUES - Kernel needs fixing!")
        print(f"   → See recommendations below")
    elif severity == "WARNING":
        print(f"   ⚠️  Some issues detected - consider kernel tuning")
    else:
        print(f"   ✓  Kernel appears well-configured")
    
    print("="*70)
    
    return {
        'length_scales': length_scales,
        'severity': severity,
        'kernel': kernel
    }


# Usage
diagnosis = diagnose_kernel(gp, X_obs, y_obs)

KERNEL DIAGNOSIS

1. KERNEL STRUCTURE:
   2.26**2 * RBF(length_scale=0.739)

   Type: Product kernel
   - Constant scale: 5.1043
   - Length scale(s): [0.73947145]

2. LENGTH SCALE ANALYSIS:
   Isotropic: 0.7395
   ✓  GOOD - Reasonable length scale

3. NOISE LEVEL:
   Alpha: 1e-06
   ✓  Reasonable noise level

4. MODEL FIT QUALITY:
   Training RMSE: 0.000068
   Training MAE:  0.000038
   Mean std dev:  0.006699
   ⚠️  Possible overfitting (too good fit)

5. HYPERPARAMETER BOUNDS:
   Length scale bounds: (0.01, 10.0)
   ✓  Not at bounds

6. OVERALL ASSESSMENT:
   ✓  Kernel appears well-configured


In [24]:
def diagnose_stagnation(gp, X_obs, y_obs, bounds, kappa=3.0):
    """
    Determine why optimization is stagnated and what to do.
    """
    print("\n" + "="*70)
    print("STAGNATION DIAGNOSTIC")
    print("="*70)
    
    dim = len(bounds)
    best_idx = np.argmax(y_obs)
    best_x = X_obs[best_idx]
    best_y = y_obs[best_idx]
    
    lows = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])
    
    # Check 1: Is the best point on boundary?
    print("\n1. BOUNDARY CHECK:")
    distances_to_lower = best_x - lows
    distances_to_upper = highs - best_x
    min_distances = np.minimum(distances_to_lower, distances_to_upper)
    
    boundary_threshold = 0.05
    on_boundary_dims = np.where(min_distances < boundary_threshold)[0]
    
    if len(on_boundary_dims) > 0:
        print(f"   ❌ PROBLEM: Best point on boundary!")
        print(f"   Dimensions: {on_boundary_dims.tolist()}")
        print(f"   Best point: {best_x}")
        for dim_idx in on_boundary_dims:
            print(f"     x{dim_idx}: {best_x[dim_idx]:.4f} (bounds: [{lows[dim_idx]:.2f}, {highs[dim_idx]:.2f}])")
        root_cause = "BOUNDARY"
        confidence = "HIGH"
    else:
        print(f"   ✓ Not on boundary")
        root_cause = None
        confidence = None
    
    # Check 2: Is uncertainty high everywhere?
    print("\n2. UNCERTAINTY CHECK:")
    _, std_at_best = gp.predict(best_x.reshape(1, -1), return_std=True)
    std_at_best = std_at_best[0]
    
    # Sample random points
    X_random = np.random.uniform(low=lows, high=highs, size=(1000, dim))
    _, std_random = gp.predict(X_random, return_std=True)
    
    print(f"   Uncertainty at best: {std_at_best:.4f}")
    print(f"   Mean uncertainty (random): {std_random.mean():.4f}")
    print(f"   Max uncertainty (random): {std_random.max():.4f}")
    
    if std_random.mean() > 0.3:
        print(f"   ❌ PROBLEM: High uncertainty everywhere")
        print(f"   → GP hasn't learned the space well")
        if root_cause is None:
            root_cause = "HIGH_UNCERTAINTY"
            confidence = "HIGH"
    elif std_at_best > 0.1:
        print(f"   ⚠️  High uncertainty at best point")
        if root_cause is None:
            root_cause = "UNCERTAIN_OPTIMUM"
            confidence = "MEDIUM"
    else:
        print(f"   ✓ Uncertainty reasonable")
    
    # Check 3: Is UCB flat (no differentiation)?
    print("\n3. UCB DIFFERENTIATION CHECK:")
    mu_random = gp.predict(X_random)
    ucb_random = mu_random + kappa * std_random
    
    ucb_range = ucb_random.max() - ucb_random.min()
    ucb_std = ucb_random.std()
    
    print(f"   UCB range: {ucb_range:.4f}")
    print(f"   UCB std dev: {ucb_std:.4f}")
    print(f"   Best UCB: {ucb_random.max():.4f}")
    print(f"   Current best y: {best_y:.4f}")
    
    if ucb_range < 0.1:
        print(f"   ❌ PROBLEM: UCB is flat (no differentiation)")
        print(f"   → Acquisition function not working")
        if root_cause is None:
            root_cause = "FLAT_UCB"
            confidence = "HIGH"
    elif ucb_random.max() < best_y + 0.01:
        print(f"   ⚠️  UCB not suggesting better regions")
        if root_cause is None:
            root_cause = "LOW_UCB"
            confidence = "MEDIUM"
    else:
        print(f"   ✓ UCB shows differentiation")
    
    # Check 4: Kernel issues
    print("\n4. KERNEL CHECK:")
    kernel = gp.kernel_
    
    # Extract length scales
    if hasattr(kernel, 'k2') and hasattr(kernel.k2, 'length_scale'):
        length_scales = np.atleast_1d(kernel.k2.length_scale)
    elif hasattr(kernel, 'length_scale'):
        length_scales = np.atleast_1d(kernel.length_scale)
    else:
        length_scales = None
    
    kernel_issue = False
    if length_scales is not None:
        print(f"   Length scales: {length_scales}")
        
        if np.any(length_scales < 0.05):
            print(f"   ❌ PROBLEM: Some length scales too short")
            kernel_issue = True
        elif np.any(length_scales > 5.0):
            print(f"   ❌ PROBLEM: Some length scales too long")
            kernel_issue = True
        else:
            print(f"   ✓ Length scales reasonable")
        
        if len(length_scales) > 1:
            ratio = length_scales.max() / length_scales.min()
            print(f"   Length scale ratio: {ratio:.2f}")
            if ratio > 100:
                print(f"   ❌ PROBLEM: Extreme anisotropy")
                kernel_issue = True
        
        if kernel_issue and root_cause is None:
            root_cause = "BAD_KERNEL"
            confidence = "HIGH"
    else:
        print(f"   ⚠️  Cannot extract length scales")
    
    # Check 5: Sampling density
    print("\n5. SAMPLING DENSITY CHECK:")
    n_obs = len(y_obs)
    samples_per_dim = n_obs / dim
    
    print(f"   Total samples: {n_obs}")
    print(f"   Samples per dimension: {samples_per_dim:.1f}")
    
    if samples_per_dim < 5:
        print(f"   ❌ PROBLEM: Very sparse sampling")
        if root_cause is None:
            root_cause = "SPARSE_SAMPLING"
            confidence = "MEDIUM"
    elif samples_per_dim < 10:
        print(f"   ⚠️  Moderate sampling density")
    else:
        print(f"   ✓ Good sampling density")
    
    # Check 6: Recent exploration pattern
    print("\n6. EXPLORATION PATTERN:")
    if n_obs >= 10:
        recent_points = X_obs[-5:]
        from scipy.spatial.distance import pdist
        
        recent_distances = pdist(recent_points)
        print(f"   Recent pairwise distances (mean): {recent_distances.mean():.4f}")
        
        if recent_distances.mean() < 0.05:
            print(f"   ❌ PROBLEM: Recent samples too clustered")
            print(f"   → Over-exploiting, not exploring")
            if root_cause is None:
                root_cause = "OVER_EXPLOITATION"
                confidence = "HIGH"
        else:
            print(f"   ✓ Recent samples spread out")
    
    # FINAL DIAGNOSIS
    print("\n" + "="*70)
    print("DIAGNOSIS:")
    print("="*70)
    
    if root_cause == "BOUNDARY":
        print(f"\n🎯 ROOT CAUSE: Optimum on boundary ({confidence} confidence)")
        print(f"\n   RECOMMENDED ACTION: Expand search space")
        print(f"   Priority: 1 (Highest)")
        action = "EXPAND_BOUNDS"
        
    elif root_cause == "BAD_KERNEL":
        print(f"\n🎯 ROOT CAUSE: Kernel misconfigured ({confidence} confidence)")
        print(f"\n   RECOMMENDED ACTION: Fix kernel")
        print(f"   Priority: 1 (Highest)")
        action = "FIX_KERNEL"
        
    elif root_cause == "OVER_EXPLOITATION" or root_cause == "FLAT_UCB":
        print(f"\n🎯 ROOT CAUSE: Insufficient exploration ({confidence} confidence)")
        print(f"\n   RECOMMENDED ACTION: Increase kappa")
        print(f"   Priority: 1 (Highest)")
        action = "INCREASE_KAPPA"
        
    elif root_cause == "HIGH_UNCERTAINTY" or root_cause == "SPARSE_SAMPLING":
        print(f"\n🎯 ROOT CAUSE: Insufficient sampling ({confidence} confidence)")
        print(f"\n   RECOMMENDED ACTION: Collect more samples")
        print(f"   Priority: 2 (Medium)")
        action = "MORE_SAMPLES"
        
    elif root_cause == "UNCERTAIN_OPTIMUM" or root_cause == "LOW_UCB":
        print(f"\n🎯 ROOT CAUSE: Need local refinement ({confidence} confidence)")
        print(f"\n   RECOMMENDED ACTION: Refine around best point")
        print(f"   Priority: 3 (Lower)")
        action = "REFINE_LOCAL"
        
    else:
        print(f"\n🤔 ROOT CAUSE: Unclear")
        print(f"\n   RECOMMENDED ACTION: Try multiple fixes")
        print(f"   Priority: -")
        action = "TRY_ALL"
    
    print("="*70)
    
    return {
        'root_cause': root_cause,
        'action': action,
        'confidence': confidence,
        'on_boundary_dims': on_boundary_dims.tolist(),
        'std_at_best': std_at_best,
        'kernel_issue': kernel_issue,
        'length_scales': length_scales.tolist() if length_scales is not None else None
    }


# RUN THE DIAGNOSTIC
diagnosis = diagnose_stagnation(gp, X_obs, y_obs, bounds, kappa=3.0)


STAGNATION DIAGNOSTIC

1. BOUNDARY CHECK:
   ✓ Not on boundary

2. UNCERTAINTY CHECK:
   Uncertainty at best: 0.0067
   Mean uncertainty (random): 1.2722
   Max uncertainty (random): 4.0787
   ❌ PROBLEM: High uncertainty everywhere
   → GP hasn't learned the space well

3. UCB DIFFERENTIATION CHECK:
   UCB range: 32.7528
   UCB std dev: 6.1464
   Best UCB: -0.3746
   Current best y: -4.0255
   ✓ UCB shows differentiation

4. KERNEL CHECK:
   Length scales: [0.73947145]
   ✓ Length scales reasonable

5. SAMPLING DENSITY CHECK:
   Total samples: 33
   Samples per dimension: 8.2
   ⚠️  Moderate sampling density

6. EXPLORATION PATTERN:
   Recent pairwise distances (mean): 0.7835
   ✓ Recent samples spread out

DIAGNOSIS:

🎯 ROOT CAUSE: Insufficient sampling (HIGH confidence)

   RECOMMENDED ACTION: Collect more samples
   Priority: 2 (Medium)
